# Контрольная 1: базовый синтаксис

**1 час · 6 задач · для завершения нужно решить 5 из 6**

### Как выполнять

1. Решайте задания самостоятельно. Конспект и справочник можно использовать.
2. Запускайте **базовые и расширенные проверки**. Ошибки можно исправлять без ограничений.
3. После собственной попытки можно открыть разбор под заданием. Если посмотрели готовое решение, закройте его и восстановите код самостоятельно перед повторной проверкой.

### Когда можно продолжать

Задача засчитывается, только если **пройдены обе группы проверок**.

После решения **5 из 6 задач** можно переходить дальше. Оставшуюся ошибку обязательно разберите.

> Отправлять работу на проверку не нужно.

## Как решать и проверять себя

### 1. Подготовьте ноутбук

Запустите служебную ячейку ниже. Это нужно делать после каждого открытия или перезапуска среды.

### 2. Напишите решение

В ячейке ответа оставьте первую строку `%%answer НОМЕР`. Под ней напишите свой код вместо заглушки `raise NotImplementedError(...)`.

В упражнении с опечаткой исправьте неправильное имя функции.

### 3. Запустите проверку

Сначала запустите ячейку со своим ответом, затем следующую ячейку с проверками.

**Ячейка ответа только сохраняет код. Проверка запускает его и сама подставляет входные данные.** Вводить что-либо вручную через `input()` не нужно.

У каждого задания две группы проверок: **базовые и расширенные**. Для завершения задания должны пройти обе.

### 4. Исправьте ошибки

Если проверка не прошла, прочитайте сообщение: оно покажет, что ожидалось и что получилось. Обозначение `\n` означает перевод строки, а `\t` означает табуляцию.

После исправления снова запустите ячейку ответа и ячейку проверки. Если изменили решение, от которого зависят следующие задания, повторите и их проверки.

### 5. Используйте подсказки

Если застряли на 10–15 минут, откройте **«Подсказку»**, а затем **«Разбор и пример решения»**.

Разберитесь в логике, закройте пример и попробуйте написать решение самостоятельно. Ваш код может отличаться от образца.

### 6. Завершите тему

Проверьте общий результат, ответьте на вопросы и заполните чек-лист. Затем перезапустите среду, выполните все ячейки сверху вниз и сохраните ноутбук.

---

### Важно помнить

- Используйте переменные из условия. Не подменяйте их готовыми числами или строками ради прохождения проверки.
- Если в условии требуется `while`, `map` или другой приём, убедитесь, что применили именно его.
- Готовые разборы даны для чтения. Команда «Выполнить все ячейки» не переносит их в ваши ответы.
- Строка `%%answer НОМЕР` работает в ноутбуке, включая JupyterLab и Colab. При переносе решения в обычный файл `.py` удалите её.

**Задание завершено, когда пройдены обе группы проверок и вы можете объяснить своё решение.**

In [ ]:
# ДАННУЮ ЯЧЕЙКУ ТОЛЬКО ЗАПУСТИТЬ, НЕОБХОДИМА ДЛЯ ЗАПУСКА ПРОВЕРОК РАБОТЫ.

"""Standalone self-check runtime, embedded in every self-study notebook."""
import ast
import io
import sys
import time
import math
import copy
import traceback
import tempfile
from pathlib import Path
from contextlib import contextmanager, redirect_stdout

answers = {}
results = {}
extended_results = {}
_dependencies = {}

def _save_answer(line, cell):
    task_id = line.strip()
    if not task_id or ' ' in task_id:
        raise ValueError('После %%answer укажите один номер задания.')
    answers[task_id] = cell
    invalidated = {task_id}
    while True:
        affected = {key for key, deps in _dependencies.items() if invalidated.intersection(deps)}
        expanded = invalidated | affected
        if expanded == invalidated:
            break
        invalidated = expanded
    for key in invalidated:
        results.pop(key, None)
        extended_results.pop(key, None)
    print(f'{task_id}: ответ сохранён. Запустите следующую ячейку — полную самопроверку.')

try:
    _shell = get_ipython()
except NameError:
    _shell = None
if _shell is not None:
    _shell.register_magic_function(_save_answer, magic_kind='cell', magic_name='answer')

def _equal(actual, expected):
    if isinstance(expected, float):
        return type(actual) in (int, float) and math.isclose(actual, expected, rel_tol=1e-9, abs_tol=1e-9)
    if type(actual) is not type(expected):
        return False
    if isinstance(expected, (list, tuple)):
        return len(actual) == len(expected) and all(_equal(a, b) for a, b in zip(actual, expected))
    if isinstance(expected, dict):
        return actual.keys() == expected.keys() and all(_equal(actual[k], expected[k]) for k in expected)
    return actual == expected

def expect(actual, expected, context='Результат'):
    if not _equal(actual, expected):
        raise AssertionError(f'{context}\nОжидалось: {expected!r}\nПолучено:  {actual!r}')

def require(condition, description):
    if not condition:
        raise AssertionError(f'Не выполнено условие: {description}')

def render_checks(source):
    """Turn bare equality assertions into actual/expected feedback; no eval."""
    class ExplainAssertions(ast.NodeTransformer):
        def visit_Assert(self, node):
            text = ast.get_source_segment(source, node.test) or ast.unparse(node.test)
            test = node.test
            if node.msg is None and isinstance(test, ast.Compare) and len(test.ops)==1 and isinstance(test.ops[0], ast.Eq):
                call = ast.Call(func=ast.Name(id='expect', ctx=ast.Load()), args=[test.left,test.comparators[0],ast.Constant('Проверка: '+text)], keywords=[])
            else:
                call = ast.Call(func=ast.Name(id='require', ctx=ast.Load()), args=[test,node.msg or ast.Constant(text)], keywords=[])
            return ast.copy_location(ast.Expr(value=call),node)
    tree=ExplainAssertions().visit(ast.parse(source))
    ast.fix_missing_locations(tree)
    return ast.unparse(tree)

@contextmanager
def captured():
    buffer=io.StringIO()
    with redirect_stdout(buffer):
        yield buffer

@contextmanager
def sample_files(contents):
    with tempfile.TemporaryDirectory(prefix='python_nlp_') as directory:
        folder=Path(directory)
        for name,text in contents.items():
            path=folder/name
            path.parent.mkdir(parents=True,exist_ok=True)
            path.write_text(text,encoding='utf-8')
        yield folder

def run(task_id, variables=None, stdin='', deps=None):
    _dependencies.setdefault(task_id,set()).update(deps or [])
    stream=io.StringIO(stdin)
    def read_input(prompt=''):
        if prompt:
            print(prompt,end='')
        line=stream.readline()
        if line=='':
            raise EOFError('Запрошено больше строк, чем дано в условии. Проверьте число input() и условие цикла.')
        return line.rstrip('\n')
    environment={'__name__':'__exercise__','input':read_input}
    environment.update(copy.deepcopy(variables or {}))
    with captured() as output:
        for key in (deps or [])+[task_id]:
            if key not in answers:
                raise NotImplementedError(f'Сначала запустите ячейку ответа {key}.')
            exec(compile(answers[key],f'<answer:{key}>','exec'),environment)
    return environment,output.getvalue()

def io_case(task_id, stdin, expected):
    _,output=run(task_id,stdin=stdin)
    expect(output,expected,f'Вход: {stdin!r}; сравниваем вывод')

def vars_case(task_id, initial, expected):
    environment,output=run(task_id,variables=initial)
    expect(output,'','В этой задаче печать не требуется')
    for name,value in expected.items():
        require(name in environment,f'создать переменную {name}')
        expect(environment[name],value,f'Вход: {initial!r}; переменная {name}')

def function_case(task_id,name,arguments,expected,deps=None):
    environment,output=run(task_id,deps=deps)
    expect(output,'','Вывод при определении функций')
    require(name in environment and callable(environment[name]),f'определить функцию {name}')
    actual_arguments=copy.deepcopy(arguments)
    before=copy.deepcopy(actual_arguments)
    with captured() as output:
        value=environment[name](*actual_arguments)
    expect(value,expected,f'Вызов: {name}{arguments!r}')
    expect(output.getvalue(),'','Нужен возвращаемый результат, без print')
    expect(actual_arguments,before,'Исходные аргументы изменились; проверьте копирование')

def namespace_case(task_id,test_source,deps=None):
    environment,output=run(task_id,deps=deps)
    expect(output,'','Вывод при определении функций')
    environment.update(expect=expect,require=require)
    exec(compile(render_checks(test_source),f'<checks:{task_id}>','exec'),environment)

ERROR_HELP={
    'AssertionError':'Сравните вход и ожидаемый результат. Проверьте границы, тип результата и изменение исходных данных.',
    'NameError':'Проверьте написание имени и наличие присваивания или определения функции. Входные переменные перечислены в условии.',
    'TypeError':'Проверьте типы аргументов, преобразования int/str и число параметров функции.',
    'ValueError':'Проверьте формат преобразуемой строки и допустимость операции для этих данных.',
    'IndexError':'Проверьте пустые данные и границы индексов: последний допустимый индекс на 1 меньше длины.',
    'KeyError':'Проверьте наличие ключа. Когда допустимо отсутствие, используйте get с нужным значением по умолчанию.',
    'ZeroDivisionError':'Обработайте нулевой знаменатель или пустую коллекцию до деления.',
    'SyntaxError':'Проверьте двоеточия, скобки, кавычки и строку перед отмеченной ошибкой.',
    'IndentationError':'Используйте одинаковые отступы; тело if, цикла и функции должно быть сдвинуто вправо.',
    'FileNotFoundError':'Проверьте рабочую папку и путь. В заданиях используйте переданный path, а не своё имя файла.',
    'TimeoutError':'Проверьте, изменяется ли счётчик и может ли условие цикла стать ложным.',
}

def check(task_id,function,extended=False):
    target=extended_results if extended else results
    deadline=time.monotonic()+8
    lines=0
    def trace(frame,event,argument):
        nonlocal lines
        if event=='line' and frame.f_code.co_filename.startswith('<answer:'):
            lines+=1
            if lines>100000 or time.monotonic()>deadline:
                raise TimeoutError('Превышен учебный лимит выполнения.')
        return trace
    old_trace=sys.gettrace()
    label='Расширенные случаи' if extended else 'Базовые примеры'
    try:
        sys.settrace(trace)
        function()
    except NotImplementedError as error:
        target[task_id]=False
        print(f'{task_id} · {label}: ещё не решено. {error}')
    except Exception as error:
        target[task_id]=False
        print(f'{task_id} · {label}: нужно исправить.\n{type(error).__name__}: {error}')
        answer_frames=[frame for frame in traceback.extract_tb(error.__traceback__) if frame.filename.startswith('<answer:')]
        if answer_frames:
            frame=answer_frames[-1]
            key=frame.filename.removeprefix('<answer:').removesuffix('>')
            source_lines=answers.get(key,'').splitlines()
            line=source_lines[frame.lineno-1] if 0<frame.lineno<=len(source_lines) else ''
            print(f'В ответе {key}, строка {frame.lineno} (без строки %%answer): {line.strip()}')
        elif isinstance(error,SyntaxError):
            print(f'Строка ответа: {error.lineno}; код: {(error.text or "").strip()}')
        help_text=ERROR_HELP.get(type(error).__name__)
        if help_text:
            print('Как исправлять:',help_text)
        print('Если застряли: откройте подсказку, затем разбор под этим заданием.')
    else:
        target[task_id]=True
        print(f'{task_id} · {label}: пройдено.')
    finally:
        sys.settrace(old_trace)

def completed(task_id):
    return results.get(task_id,False) and extended_results.get(task_id,False)

def summary(task_ids,threshold=None):
    basic=sum(results.get(key,False) for key in task_ids)
    extended=sum(extended_results.get(key,False) for key in task_ids)
    full=sum(completed(key) for key in task_ids)
    required=len(task_ids) if threshold is None else threshold
    print(f'Базовые примеры: {basic}/{len(task_ids)}; расширенные случаи: {extended}/{len(task_ids)}.')
    print(f'Полностью проверено: {full}/{len(task_ids)}.')
    pending=[key for key in task_ids if not completed(key)]
    if pending:
        print('Вернуться к заданиям:',', '.join(pending))
    if full>=required:
        print(f'Порог автопроверки достигнут ({required}/{len(task_ids)}). Теперь ответьте на вопросы и сверьтесь с разбором.')
    else:
        print(f'Для завершения нужно {required} полностью проверенных заданий. Можно исправлять и запускать проверки сколько угодно раз.')
    print('После правок: Restart Kernel → Run All, затем сохранить. Все решения и объяснения доступны в этом ноутбуке.')
    return {'basic':basic,'extended':extended,'complete':full,'required':required,'automatic_pass':full>=required}

print('Самопроверка готова. Служебный код менять не нужно.')


## 01. Пакеты сообщений

Напишите функцию `batch_count(total, size)`, где:

- `total >= 0` количество сообщений;
- `size > 0` вместимость одного пакета.

Верните **минимальное число пакетов для всех сообщений**, учитывая последний неполный пакет.

> Если явно не сказано обратное, входные объекты менять нельзя.

In [ ]:
%%answer C1_01


def batch_count(total, size):
    # Ваш код
    raise NotImplementedError("Допишите функцию")

In [ ]:
# Полная самопроверка C1_01: запустите эту ячейку после своего ответа.

def _check_C1_01_basic():
    function_case('C1_01', 'batch_count', (25, 8), 4, deps=[])
    function_case('C1_01', 'batch_count', (0, 8), 0, deps=[])

check('C1_01', _check_C1_01_basic, extended=False)

def _check_C1_01_extended():
    function_case('C1_01', 'batch_count', (24, 8), 3, deps=[])
    function_case('C1_01', 'batch_count', (1, 8), 1, deps=[])

check('C1_01', _check_C1_01_extended, extended=True)

print("Задание полностью проверено." if completed('C1_01') else "Нужно пройти обе группы. Подсказка и разбор находятся ниже.")

<details>
<summary>Разбор и пример решения — откройте после собственной попытки</summary>

**Почему это работает.** Добавление size-1 перед целочисленным делением реализует округление вверх. Проверьте нулевое число сообщений, точное деление и единственное сообщение.

```python
def batch_count(total,size):
    return (total+size-1)//size
```

**После сравнения:** закройте блок, объясните ключевой шаг своими словами и запустите полную проверку своего ответа. При требовании использовать определённую конструкцию проверьте и этот пункт условия.

</details>

## 02. Статус сообщения

Напишите функцию `message_status(text)`, которая **возвращает строку**:

- `"missing"`, если `text` равно `None`;
- `"empty"`, если после `strip()` строка пуста;
- `"short"`, если длина очищенного текста меньше `5`;
- `"ok"` в остальных случаях.

Регистр текста не меняйте.

> Если явно не сказано обратное, входные объекты менять нельзя.

In [ ]:
%%answer C1_02


def message_status(text):
    # Ваш код
    raise NotImplementedError("Допишите функцию")

In [ ]:
# Полная самопроверка C1_02: запустите эту ячейку после своего ответа.

def _check_C1_02_basic():
    function_case('C1_02', 'message_status', (None,), 'missing', deps=[])
    function_case('C1_02', 'message_status', ('  ',), 'empty', deps=[])
    function_case('C1_02', 'message_status', ('abcd',), 'short', deps=[])

check('C1_02', _check_C1_02_basic, extended=False)

def _check_C1_02_extended():
    function_case('C1_02', 'message_status', (' abcde ',), 'ok', deps=[])
    function_case('C1_02', 'message_status', ('',), 'empty', deps=[])

    function_case('C1_02', 'message_status', ('  ab  ',), 'short', deps=[])

check('C1_02', _check_C1_02_extended, extended=True)

print("Задание полностью проверено." if completed('C1_02') else "Нужно пройти обе группы. Подсказка и разбор находятся ниже.")

<details>
<summary>Разбор и пример решения — откройте после собственной попытки</summary>

**Почему это работает.** None проверяется до строковых методов. Затем strip убирает края, а длина сравнивается уже у очищенного текста; длина ровно 5 относится к ok.

```python
def message_status(text):
    if text is None:
        return "missing"
    cleaned=text.strip()
    if not cleaned:
        return "empty"
    if len(cleaned)<5:
        return "short"
    return "ok"
```

**После сравнения:** закройте блок, объясните ключевой шаг своими словами и запустите полную проверку своего ответа. При требовании использовать определённую конструкцию проверьте и этот пункт условия.

</details>

## 03. Количество положительных до нуля

Напишите функцию `positive_before_zero(numbers)`, которая **возвращает количество положительных чисел до первого нуля**.

Элементы после первого нуля не учитывайте. Если нуля нет, проверьте весь список.

> Если явно не сказано обратное, входные объекты менять нельзя.

In [ ]:
%%answer C1_03


def positive_before_zero(numbers):
    # Ваш код
    raise NotImplementedError("Допишите функцию")

In [ ]:
# Полная самопроверка C1_03: запустите эту ячейку после своего ответа.

def _check_C1_03_basic():
    function_case('C1_03', 'positive_before_zero', ([1, -2, 3, 0, 9],), 2, deps=[])
    function_case('C1_03', 'positive_before_zero', ([],), 0, deps=[])

check('C1_03', _check_C1_03_basic, extended=False)

def _check_C1_03_extended():
    function_case('C1_03', 'positive_before_zero', ([0, 1],), 0, deps=[])
    function_case('C1_03', 'positive_before_zero', ([-1, 2],), 1, deps=[])

    function_case('C1_03', 'positive_before_zero', ([2, 3, 0],), 2, deps=[])
    function_case('C1_03', 'positive_before_zero', ([5, -2, 0],), 1, deps=[])
    function_case('C1_03', 'positive_before_zero', ([-3, -2, 0, 7],), 0, deps=[])
    function_case('C1_03', 'positive_before_zero', ([2, 5, 8],), 3, deps=[])
    function_case('C1_03', 'positive_before_zero', ([1, 0, 2, 3, 0, 4],), 1, deps=[])
    function_case('C1_03', 'positive_before_zero', ([-4, -2],), 0, deps=[])

check('C1_03', _check_C1_03_extended, extended=True)

print("Задание полностью проверено." if completed('C1_03') else "Нужно пройти обе группы. Подсказка и разбор находятся ниже.")

<details>
<summary>Разбор и пример решения — откройте после собственной попытки</summary>

**Почему это работает.** Ноль завершает цикл раньше остальных проверок. Отрицательные значения пропускаются, положительные увеличивают счётчик только до первого нуля.

```python
def positive_before_zero(numbers):
    result=0
    for number in numbers:
        if number==0:
            break
        if number>0:
            result+=1
    return result
```

**После сравнения:** закройте блок, объясните ключевой шаг своими словами и запустите полную проверку своего ответа. При требовании использовать определённую конструкцию проверьте и этот пункт условия.

</details>

## 04. Сократить строку

Напишите функцию `preview(text, limit)`, где `limit >= 0`.

Если длина `text` **больше `limit`**, верните первые `limit` символов, добавив в конец `"..."`.

Иначе верните исходную строку.

> Если явно не сказано обратное, входные объекты менять нельзя.

In [ ]:
%%answer C1_04


def preview(text, limit):
    # Ваш код
    raise NotImplementedError("Допишите функцию")

In [ ]:
# Полная самопроверка C1_04: запустите эту ячейку после своего ответа.

def _check_C1_04_basic():
    function_case('C1_04', 'preview', ('abcdef', 3), 'abc...', deps=[])
    function_case('C1_04', 'preview', ('abc', 3), 'abc', deps=[])

check('C1_04', _check_C1_04_basic, extended=False)

def _check_C1_04_extended():
    function_case('C1_04', 'preview', ('x', 0), '...', deps=[])
    function_case('C1_04', 'preview', ('', 0), '', deps=[])

check('C1_04', _check_C1_04_extended, extended=True)

print("Задание полностью проверено." if completed('C1_04') else "Нужно пройти обе группы. Подсказка и разбор находятся ниже.")

<details>
<summary>Разбор и пример решения — откройте после собственной попытки</summary>

**Почему это работает.** Многоточие добавляется только при строгом превышении limit. При нулевом limit непустая строка превращается в "...", а пустая остаётся пустой.

```python
def preview(text,limit):
    if len(text)>limit:
        return text[:limit]+"..."
    return text
```

**После сравнения:** закройте блок, объясните ключевой шаг своими словами и запустите полную проверку своего ответа. При требовании использовать определённую конструкцию проверьте и этот пункт условия.

</details>

## 05. Удалить короткие слова

Напишите функцию `keep_long(words, minimum)`, где `minimum >= 0`.

Верните **новый список слов длиной не меньше `minimum`**. Сохраните исходный порядок и повторы.

> Если явно не сказано обратное, входные объекты менять нельзя.

In [ ]:
%%answer C1_05


def keep_long(words, minimum):
    # Ваш код
    raise NotImplementedError("Допишите функцию")

In [ ]:
# Полная самопроверка C1_05: запустите эту ячейку после своего ответа.

def _check_C1_05_basic():
    function_case('C1_05', 'keep_long', (['я', 'кот', 'кот'], 3), ['кот', 'кот'], deps=[])
    function_case('C1_05', 'keep_long', ([], 1), [], deps=[])

check('C1_05', _check_C1_05_basic, extended=False)

def _check_C1_05_extended():
    function_case('C1_05', 'keep_long', (['', 'а'], 0), ['', 'а'], deps=[])

    function_case('C1_05', 'keep_long', (['пёс', 'кот', 'дом'], 3), ['пёс', 'кот', 'дом'], deps=[])

check('C1_05', _check_C1_05_extended, extended=True)

print("Задание полностью проверено." if completed('C1_05') else "Нужно пройти обе группы. Подсказка и разбор находятся ниже.")

<details>
<summary>Разбор и пример решения — откройте после собственной попытки</summary>

**Почему это работает.** Списковое включение отбирает элементы по длине и сохраняет порядок. Нельзя превращать результат в множество: повторяющиеся подходящие слова нужны.

```python
def keep_long(words,minimum):
    return [word for word in words if len(word)>=minimum]
```

**После сравнения:** закройте блок, объясните ключевой шаг своими словами и запустите полную проверку своего ответа. При требовании использовать определённую конструкцию проверьте и этот пункт условия.

</details>

## 06. Частота символа

Напишите функцию `char_share(text, char)`, где `char` содержит один символ.

Верните **долю его вхождений в `text`**: количество вхождений, делённое на длину строки. Регистр учитывается.

Для пустой строки верните `0.0`.

> Если явно не сказано обратное, входные объекты менять нельзя.

In [ ]:
%%answer C1_06


def char_share(text, char):
    # Ваш код
    raise NotImplementedError("Допишите функцию")

In [ ]:
# Полная самопроверка C1_06: запустите эту ячейку после своего ответа.

def _check_C1_06_basic():
    function_case('C1_06', 'char_share', ('абба', 'а'), 0.5, deps=[])
    function_case('C1_06', 'char_share', ('', 'а'), 0.0, deps=[])

check('C1_06', _check_C1_06_basic, extended=False)

def _check_C1_06_extended():
    function_case('C1_06', 'char_share', ('Аа', 'а'), 0.5, deps=[])
    function_case('C1_06', 'char_share', ('aaa', 'a'), 1.0, deps=[])

check('C1_06', _check_C1_06_extended, extended=True)

print("Задание полностью проверено." if completed('C1_06') else "Нужно пройти обе группы. Подсказка и разбор находятся ниже.")

<details>
<summary>Разбор и пример решения — откройте после собственной попытки</summary>

**Почему это работает.** Пустая строка требует отдельного результата до деления. Для остальных строк доля равна числу вхождений символа, делённому на общую длину.

```python
def char_share(text,char):
    if not text:
        return 0.0
    return text.count(char)/len(text)
```

**После сравнения:** закройте блок, объясните ключевой шаг своими словами и запустите полную проверку своего ответа. При требовании использовать определённую конструкцию проверьте и этот пункт условия.

</details>

## Итог самопроверки

Этот счётчик учитывает обе группы. Перед завершением выполните ноутбук с начала после перезапуска ядра.

In [ ]:
progress = summary(['C1_01', 'C1_02', 'C1_03', 'C1_04', 'C1_05', 'C1_06'], threshold=5)

## Проверьте понимание

Сначала ответьте своими словами в следующей текстовой ячейке. Затем раскройте ответы и исправьте неточности. Эти ответы не оцениваются автоматически.

1. Как проверить правильность решения без совпадения текста с разбором?
2. Почему print(result) не заменяет return result в этих задачах?

Мои ответы:

1. …
2. …

<details>
<summary>Ответы для сравнения</summary>

**1. Как проверить правильность решения без совпадения текста с разбором?**

Сравните контракт: входные типы, возвращаемый результат, пустые и граничные случаи, отсутствие запрещённых изменений. Разные алгоритмы могут быть одинаково верны.

**2. Почему print(result) не заменяет return result в этих задачах?**

Проверки вызывают функции и сравнивают полученное значение. print выводит текст, но без return функция возвращает None.

</details>

## Личный чек-лист завершения

После сверки замените нужные `False` на `True`. Это ваша оценка понимания: программа не умеет автоматически оценивать свободное объяснение. Если пока не можете подтвердить пункт, вернитесь к соответствующему заданию или разбору.

In [ ]:
self_review = {
    "Я могу объяснить ключевые шаги своих решений": False,
    "Я сверил требуемые конструкции и ответы на вопросы": False,
    "После разбора я могу воспроизвести решение без подсматривания": False,
}
understanding_checked = all(self_review.values())
automatic_ready = sum(completed(key) for key in ['C1_01', 'C1_02', 'C1_03', 'C1_04', 'C1_05', 'C1_06']) >= 5
module_complete = automatic_ready and understanding_checked
if module_complete:
    print("Модуль завершён по тестам и вашему чек-листу. Сохраните ноутбук; можно продолжать маршрут.")
elif not automatic_ready:
    print("Сначала завершите полную автопроверку. После исправлений повторите Restart Kernel → Run All.")
else:
    print("Автопроверка пройдена. Осталось сверить понимание и отметить личный чек-лист.")